In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import os
import time

# Fecha y hora actual
now = datetime.now()

# URL principal
url = "https://www.cooperativa.cl/noticias/regiones/region-de-valparaiso"

# Lista para guardar URLs únicas
lista_url = []

# Lista para guardar las noticias
noticias = []

# Pedimos la página principal
respuesta = requests.get(url)

if respuesta.status_code == 200:
    soup_principal = BeautifulSoup(respuesta.text, "html.parser")

    # Buscar todos los enlaces
    enlaces = soup_principal.find_all("a", href=True)

    for enlace in enlaces:
        link = enlace["href"]

        # Filtrar solo noticias
        if link.startswith("https://www.cooperativa.cl/noticias"):

            # Evitar noticias repetidas
            if link not in lista_url:
                lista_url.append(link)

                print("Extrayendo:", link)

                # Entrar a la noticia
                respuesta_noticia = requests.get(link)

                if respuesta_noticia.status_code == 200:
                    soup_noticia = BeautifulSoup(respuesta_noticia.text, "html.parser")

                    # Extraer datos
                    titulo = soup_noticia.find("h1")
                    autores = soup_noticia.find("div", class_="autor")
                    resumen = soup_noticia.find("div", class_="bajada")
                    fecha = soup_noticia.find("time", class_="fecha")
                    categorias = soup_noticia.find("div", class_="articulo-categorias")

                    # Validar si existen los datos
                    titulo_texto = titulo.get_text(strip=True) if titulo else "Sin título"
                    autores_texto = autores.get_text(strip=True) if autores else "Sin autor"
                    resumen_texto = resumen.get_text(strip=True) if resumen else "Sin resumen"
                    fecha_texto = fecha.get_text(strip=True) if fecha else "Sin fecha"

                    # Extraer etiquetas
                    if categorias:
                        etiquetas = categorias.find_all("a")
                        etiquetas_texto = [e.get_text(strip=True) for e in etiquetas]
                    else:
                        etiquetas_texto = []

                    # Guardar noticia
                    noticias.append({
                        "Titulo": titulo_texto,
                        "Autores": autores_texto,
                        "Resumen": resumen_texto,
                        "Fecha": fecha_texto,
                        "Etiquetas": etiquetas_texto,
                        "URL": link
                    })

                    # Pausa pequeña para no hacer muchas peticiones seguidas
                    time.sleep(1)

else:
    print("Error al acceder a la página:", respuesta.status_code)


# Convertir lista de noticias a DataFrame
datos = pd.DataFrame(noticias)

# Crear carpeta con fecha actual
fecha_actual = now.strftime("%Y%m%d")
carpeta = f"DATA/{fecha_actual}"

os.makedirs(carpeta, exist_ok=True)

# Guardar archivo CSV
ruta_csv = f"{carpeta}/noticias.csv"
datos.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

# Crear o actualizar archivo log
with open("registro.log", "a", encoding="utf-8") as archivo_log:
    archivo_log.write(
        f"{now} - {len(lista_url)} noticias extraídas con éxito - "
        f"Sitio web: {url}\n"
    )

print("Fin del proceso")
print("Total de noticias extraídas:", len(lista_url))

Extrayendo: https://www.cooperativa.cl/noticias/stat/clima/port/clima.html
Extrayendo: https://www.cooperativa.cl/noticias/deportes
Extrayendo: https://www.cooperativa.cl/noticias/regiones
Extrayendo: https://www.cooperativa.cl/noticias/site/tax/port/all/rss____1.xml


C:\Users\edwin\anaconda3\lib\site-packages\bs4\builder\__init__.py:545: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  warnings.warn(


Fin del proceso
Total de noticias extraídas: 4
